In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as  plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.api import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error




In [ ]:
dataFrame = pd.read_csv('Homicidios_2014_2015\Serie_homicidios_Ceara_2014-2025.csv', sep=';')

In [ ]:
dataFrame.head()

In [ ]:
dataFrame.drop(columns=['M'], inplace=True)
dataFrame.drop(columns=['F'], inplace=True)

In [ ]:
dataFrame.head()

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(dataFrame['MES'], dataFrame['CVLI'])
plt.title('Homícidios no Ceará')
plt.xlabel('Meses')
plt.ylabel('Número de Mortes Ocorridas')

#Definindo o intervalo menor para os anos não ficarem sobrepostos
id = np.linspace(0, len(dataFrame) - 1, 5, dtype=int)

meses_selecionados = dataFrame['MES'].iloc[id]

plt.xticks(meses_selecionados, rotation=0)

plt.show()

In [ ]:
#Criando uma decomposição de Séries temporais com modelo aditivo em um periodo de 12 meses 

decomposicao = seasonal_decompose(dataFrame['CVLI'], model='additive', period=12)

In [ ]:
figura_serie = decomposicao.plot()
figura_serie.set_size_inches(12, 8)
plt.show()

CVLI - (Crimes Violentos Letais e Intencionais) estado do Ceará 2014 - 2025  

*  VARIAÇÃO: Os dados de CVLI iniciam com o patamar um pouco superior a 400 no ano de 2014, seguido de algumas oscilações. Posteriormente, observar-se uma redução drastica entre os meses finais de 2018 e iniciais de 2019. No final de 2017 foi observado o maior valor de CVLIs da série no estado, onde em 2017 foi o ano com maior valor acumulado nesse retrato de 12 anos. Em contrada partida, no ano de 2019 apresentou os menores valores mensais observados na série, onde os valores mensais giraram em torno de 200 CVLI.

* Volatilidade: Observa-se vários dados com fortes oscilações, com picos muitos acentuados e reduções drasticas am alguns periodos da serie.Essas flutuações dos dados podem ter sido influênciadas por eventos externos como crises na segurança pública, conflitos entre facções criminosas por dominancia de territórios para o tráfico de drogas e entre outros eventos externos.  


Trend (Tendência)
*  Queda: A série inicia  com valor de 400 CVLI no ano de 2014, seguida de uma queda suave até atingir seu menor valor no mês 35 (Fim do ano de 2015). Posteriormente, houve um aumento expressivo atingindo o seu pico no mês 45 (ano de 2017) o ano mais violento observado em todo o intervalo.
Nos anos seguintes houve queda acentuada nos números de CVLI até o mês 65 (Ano de 2019).   

* Estabilidade: Após o mês 80 (ano de 2020), a têndencia dos dados é queda suave e constante, sugerindo que politicas públicas de segurança ou condições externas (Fim de brigas entre facções por influência em território de tráfico de drogas) podem ter afetado os números de CVLI, onde os números de assassinatos estiveram sob relativo controle nos últimos anos da séries, sem novos picos explosivos observados.

Seasonal (Sazonal)
*  Observa-se uma padrão sazonal claro nos homicidios do estado do Ceará ao longo dos anos. O primeiro período de aumento ocorre entre os meses de fevereiro e março, com expressivo aumento dos assassinatos, possivelmente influenciado por festividades como carnaval e o período de ferias.

* O maior pico sazonal é observado entre os mese de Julho e Agosto, concidindo com férias do meio do ano, perído em que se registra as maiores altas nos crimes no estado. Em seguida, observa-se um período de estabilidade entre os meses de agosto e setembro, com pouca flutuações nos números.

* Por fim, após essa estabilidade há uma queda centuada e contínua, que culmina em uma redução considerável atingindo seu ponto mais baixo no mês de dezembro.

Resid (Residula/Irregular/Restante)
*   Aleatoriedade: a maioria dos pontos concentra-se em torno de zero, sugerindo que o modelo de decomposição copturou bem os padrões(tendência e sazonalidade) da série. Os residuos aparentam ser aleatórios, sem viés evidente.  

* Interpertação: Outliers (Pontos fora da curva) observa um outliers consideravel no mês 72 (ano de 2020). Neste ano houve uma crise policial, onde delegacias foram fechadas e parte da força policial do estado cruzou os braços e não foi as ruas. Eventos extradoinarios como esse não são capturados pelos componetes de tendência e sazonalidade.

In [ ]:
#Utilizando o o modelo Exponetial Smoothing (Suavização Exponencial)
#É um modelo simples e de fácil implementação que atribui pesos maiores às observações mais recentes. 

modelo_suavizacao_exponencial = ExponentialSmoothing(
    dataFrame['CVLI'], 
    trend=None,
    seasonal='add',
    seasonal_periods=12
)

In [ ]:
fit_modelo = modelo_suavizacao_exponencial.fit()

In [ ]:
#Previsões futuras 
forecast = fit_modelo.forecast(24)

In [ ]:
plt.figure(figsize=(14, 7))

In [ ]:
#Verificando os residuos do modelo 
fit_modelo.resid.plot(title='Resíduos do Modelo', figsize=(14, 7))
plt.show()

In [ ]:
print(acorr_ljungbox(fit_modelo.resid, lags=[10]))

In [ ]:
plot_acf(fit_modelo.resid, lags=20)
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(forecast, label='Previsão Homicidios', linestyle='--')

In [ ]:
df_previsao_smoothing_exponential = pd.DataFrame({
    'Meses Projetados' : forecast.index, 
    'Estimativa': forecast.values.round(0)
})

In [ ]:
print(df_previsao_smoothing_exponential)

In [ ]:
#Crianod um modelo e definindo o tamanho do teste (24 meses)

tamanho_teste = 24
treino = dataFrame.iloc[:-tamanho_teste]
teste = dataFrame.iloc[-tamanho_teste:]


print(f"Treinando com dados até: {treino['MES'].iloc[-1]}")
print(f"Testando com dados de: {teste['MES'].iloc[0]} até {teste['MES'].iloc[-1]}")

In [ ]:
#Criando o modelo 

modelo_validacao = ExponentialSmoothing(
    treino['CVLI'], 
    trend=None, 
    seasonal='add',
    seasonal_periods=24
).fit()

In [ ]:
#Prevendo os valores para o periodo que foi separado para teste 

previsoes_teste = modelo_validacao.forecast(24)

In [ ]:
MAE = mean_absolute_error(teste['CVLI'], previsoes_teste)
MAPE = mean_absolute_percentage_error(teste['CVLI'], previsoes_teste) * 100 

print(f"Erro médio absoluto (MAE): {MAE:.2f} homicidios")
print(f"Erro percentual (MAPE):  {MAPE:.2f}%")

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(treino['MES'], treino['CVLI'], label='Treino (Histórico)')
plt.plot(teste['MES'], teste['CVLI'], label='Teste (Realidade)', color='green')
plt.plot(teste['MES'], previsoes_teste, label='Previsão do Modelo', color='red', linestyle='--')

plt.title('Validação do Modelo: Real vs Previsto')
plt.legend()

#Definindo o intervalo de 5 para exibir os dados
indices = np.linspace(0, len(dataFrame) - 1, 5, dtype=int)

ticks_selecionados = dataFrame['MES'].iloc[indices]

plt.xticks(ticks_selecionados, rotation=0)

#Rotacioando os dados na eixo x
plt.xticks(rotation=45)

plt.show()

In [ ]:
df_previsao = pd.DataFrame({
    'Meses Projetado': forecast.index,
    'Estimativa_CVLI': forecast.values.round(0)
})

print(df_previsao)